# CLIPPER
A free alternative to `opus.pro` and `vidyo.ai`

**Supported AI Backends:** Gemini · DeepSeek · Qwen · G4F (free) · Local GGUF · Manual

[![](https://dcbadge.limes.pink/api/server/tAdPHFAbud)](https://discord.gg/tAdPHFAbud)

In [ ]:
#@title ⚙️ Step 1 — API Keys Setup
# Add your API keys to Colab Secrets (🔑 icon in the left sidebar)
# and enable notebook access for each one.
# Supported secret names:
#   GEMINI_API_KEY   — Google Gemini
#   DEEPSEEK_API_KEY — DeepSeek
#   QWEN_API_KEY     — Alibaba Qwen

import os
try:
    from google.colab import userdata
    for key in ('GEMINI_API_KEY', 'DEEPSEEK_API_KEY', 'QWEN_API_KEY'):
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f'  ✅ {key} loaded')
        except Exception:
            pass
except ImportError:
    pass

print('\nKeys ready. You can also paste them directly in the UI.')

In [ ]:
#@title 🛠️ Step 2 — Installation
import os
import subprocess
import shutil
from IPython.display import clear_output

# 1. Clean previous installation
print('Cleaning previous installation...')
%cd /content
if os.path.exists('CLIPPER'):
    shutil.rmtree('CLIPPER')

!git clone https://github.com/zakiach555/CLIPPER.git
%cd /content/CLIPPER

# 2. Install UV and system deps
print('Installing UV and system drivers...')
subprocess.run(['pip', 'install', 'uv'], check=True)
subprocess.run('sudo apt update -y && sudo apt install -y libcudnn8 ffmpeg xvfb', shell=True, check=True)

# 3. Create virtual environment
print('Creating virtual environment...')
subprocess.run(['uv', 'venv', '.venv'], check=True)

# 4. Install dependencies
print('Installing libraries...')

# Phase A: WhisperX + requirements (includes openai for DeepSeek/Qwen)
for cmd in [
    'uv pip install --python .venv git+https://github.com/m-bain/whisperx.git',
    'uv pip install --python .venv -r requirements.txt',
    'uv pip install --python .venv yt-dlp pytubefix',
    'uv pip install yt_dlp',
]:
    subprocess.run(cmd, shell=True, check=True)

# Phase B: AI SDKs + alignment fix
print('Installing AI SDKs and alignment fix...')
for cmd in [
    'uv pip install --python .venv google-generativeai',
    'uv pip install --python .venv openai',          # DeepSeek + Qwen
    'uv pip install --python .venv pandas',
    'uv pip install --python .venv onnxruntime-gpu',
    'uv pip install --python .venv transformers==4.46.3 accelerate>=0.26.0',
]:
    subprocess.run(cmd, shell=True, check=True)

# Phase C: Force stable Torch 2.3.1
print('Forcing stable Torch version (2.3.1)...')
subprocess.run(
    'uv pip install --python .venv '
    'torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 '
    '--index-url https://download.pytorch.org/whl/cu121',
    shell=True, check=True
)

# Phase D: Lock NumPy
subprocess.run("uv pip install --python .venv 'numpy<2.0' setuptools==69.5.1", shell=True, check=True)

# Phase E: InsightFace + MediaPipe
print('Fixing InsightFace and MediaPipe...')
subprocess.run('uv pip install --python .venv insightface onnxruntime-gpu', shell=True, check=True)
subprocess.run('uv pip uninstall --python .venv mediapipe protobuf flatbuffers', shell=True)
subprocess.run("uv pip install --python .venv 'mediapipe>=0.10.0' 'protobuf>=3.20,<5.0' 'flatbuffers>=2.0'", shell=True, check=True)

# 5. Start virtual display
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'

clear_output()
print('✅ Installation complete!')
print('  Transformers 4.46.3 (alignment compatible): OK')
print('  Torch 2.3.1+cu121: OK')
print('  DeepSeek / Qwen (openai SDK): OK')

In [ ]:
#@title 🚀 Step 3 — Launch
import os

%cd /content/CLIPPER

# Virtual display
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'
os.environ['MPLBACKEND'] = 'Agg'

print('🚀 Starting CLIPPER...')
print('⚠️  Wait for the public Gradio URL before using the app.')
print()
print('AI Backends available: Gemini | DeepSeek | Qwen | G4F | Local | Manual')

!/content/CLIPPER/.venv/bin/python webui/app.py --colab

---
## Credits
Inspired by [reels clips automator](https://github.com/eddieoz/reels-clips-automator) and [YoutubeVideoToAIPoweredShorts](https://github.com/Fitsbit/YoutubeVideoToAIPoweredShorts)

`v0.2 Alpha`